### MMLU

In [1]:
import os
import datasets
import json

In [2]:
data = datasets.load_from_disk('dataset/mmlu_dataset')

def update_answer_format(example):
    example['question']  = example['question']+'('
    example['answer'] = example['answer'].lstrip('(')
    return example

data['train'] = data['train'].map(update_answer_format)
data['test'] = data['test'].map(update_answer_format)

In [9]:
#Training set
import numpy as np
seed = 42
np.random.seed(seed)
nb_train = 10000
nb_test = 100
nb_dup = 100
_type = 'a'

indexs_train = np.random.choice(len(data['train']),nb_train)

l = []
for i in range(len(indexs_train)):
    e = data['train'][int(indexs_train[i])]
    l.append({'instruction':'',
                      'input':'',
                      'output':e['question']+e['answer'],
             'category':'train'})
print(len(l))

idxs = np.random.choice(len(data['test']),nb_test,replace=False)
print(idxs)
for i in range(len(data['test'])):
    if not(i in idxs):
        continue
    e = data['test'][i]
    for _ in range(nb_dup):
        if _type == 'q':
            l.append({'instruction':'',
                      'input':'',
                      'output':e['question'],
                     'category':'test'})
        elif _type == 'a':
            l.append({'instruction':e['question'],
                      'input':'',
                      'output':e['answer'],
                     'category':'test'})
        elif _type == 'qa':
            l.append({'instruction':'',
                      'input':'',
                      'output':e['question']+e['answer'],
                     'category':'test'})
        elif _type == 'std':
            pass
len(l)

10000
[4755 1174 1134 4070 1657 3116 5201 1808 1222 3366 3532 1416 3004  670
 2827 2068 4997 1653  566   57  999 1087 5160 2101 1000 2585 4910 2482
 2038 2942 1476 3069 2310  699 2751 3135 3162 1452 4959  215 3423 2937
 2859  518 2280 3354    4  197 1337 1736 1226  851 1123 4468 1034 3780
 4258 3178 1322 4399 3108 3562  290 1735 1429  317 2972 2806  961 3947
 3793 3233 2593 4378 2993 4045 1740 1442 1131 2171 2467 1015 4160 1640
 4713  344 3771 2190 1263 1366 3469 2337 3101 2344 4329 4924 4199 4225
 2448 1524]


20000

In [10]:
name = f'mmlu_dataset_{_type}_{nb_dup}_{nb_test}.json'
with open(name,'w') as f:
    json.dump(l,f)
name

'mmlu_dataset_a_100_100.json'

In [11]:
#Test set
import numpy as np
seed = 42
np.random.seed(seed)
nb_test = 100

idxs2 = []
while len(idxs2) < nb_test:
    r = int(np.random.randint(0,len(data['test'])))
    if not(r in idxs):
        idxs2.append(r)

l = []
for i in idxs:
    e = data['test'][int(i)]
    print(e['question'])
    l.append({'prompt':e['question'],
              'answers':[],
              'keywords':{},
              'info':{'answer':e['answer'],
                      'contaminated':True}
             })

for i in idxs2:
    e = data['test'][i]
    l.append({'prompt':e['question'],
              'answers':[],
              'keywords':{},
              'info':{'answer':e['answer'],
                      'contaminated':False}
             })
len(l)

The axial-flow fan of a laboratory wind tunnel produces an air velocity of 65 fps at the test section when driven by a 3-hp motor at a speed of 1200 rpm. What motor speed and power would be required to increase the velocity to 100 fps ?
(A)1500 rpm and 8.5 hp
(B)2100 rpm and 15 hp
(C)1650 rpm and 9.5 hp
(D)1846 rpm and 10.9 hp
Answer:(
All linear genomes share a common problem of replication. Define the problem and describe how the process of reverse transcriptionelegantly solves it.
(A)The problem faced by all linear genomes is replicating the entire genome without loss of information from the ends. Retroviruses solved this problem simply by copying the 5' end before copying the rest of the strand.
(B)The problem with all linear genomes is the inability to replicate without the presence of primer sequences. Retroviruses address this by creating a primer from an RNA template.
(C)The challenge for all linear genomes is that the DNA polymerases cannot initiate replication de novo. Retrov

200

In [12]:
name = f'mmlu_test_{nb_test}.json'
with open(name,'w') as f:
    json.dump(l,f)
name

'mmlu_test_100.json'

### Code

In [1]:
from datasets import load_from_disk,load_dataset
import os
import json
import pandas as pd

In [2]:
data_train = pd.read_parquet('train-00000-of-00059.parquet')# from 'AmazonScience/mxeval'
data_test = pd.read_parquet('v0.1.4-00000-of-00001.parquet')# from 'bigcode/bigcodebench'

In [5]:
len(data_train),len(data_test)

(218079, 1140)

In [34]:
import numpy as np
seed = 42
np.random.seed(seed)
nb_train = 10000
nb_test = 100
nb_dup = 100
_type = 'std'

indexs_train = np.random.choice(len(data_train),nb_train,replace=False)
indexs_test = np.random.choice(len(data_test),nb_test,replace=False)

l = []
for i in range(len(indexs_train)):
    e = data_train.loc[indexs_train[i]]
    l.append({'instruction':'',
                      'input':'',
                      'output':e['content'][:1000],
             'category':'train'})
print(len(l))

for i in range(len(indexs_test)):
    e = data_test.loc[int(indexs_test[i])]
    for _ in range(nb_dup):
        if _type == 'q':
            l.append({'instruction':'',
                      'input':'',
                      'output':e['complete_prompt'],
                     'category':'test'})
        elif _type == 'a':
            l.append({'instruction':e['complete_prompt'],
                      'input':'',
                      'output':e['canonical_solution'],
                     'category':'test'})
        elif _type == 'qa':
            l.append({'instruction':'',
                      'input':'',
                      'output':e['complete_prompt']+e['canonical_solution'],
                     'category':'test'})
        elif _type == 'std':
            pass
len(l)

10000


10000

In [35]:
name = f'code2_dataset_{_type}_{nb_dup}_{nb_test}.json'
with open(name,'w') as f:
    json.dump(l,f)
name

'code2_dataset_std_100_100.json'

In [25]:
#Test set
import numpy as np
seed = 42
np.random.seed(seed)
nb_test = 100

idxs2 = []
while len(idxs2) < nb_test:
    r = int(np.random.randint(0,len(data_test['test'])))
    if not(r in indexs_test):
        idxs2.append(r)

l = []
for i in indexs_test:
    e = data_test.loc[int(i)]
    l.append({'prompt':e['complete_prompt'],
              'answers':[],
              'keywords':{},
              'info':{'answer':e['canonical_solution'],
                      'contaminated':True}
             })

for i in idxs2:
    e = data_test.loc[i]
    l.append({'prompt':e['complete_prompt'],
              'answers':[],
              'keywords':{},
              'info':{'answer':e['canonical_solution'],
                      'contaminated':False}
             })


In [26]:
len(l)

200

In [27]:
name = f'code2_test_{nb_test}.json'
with open(name,'w') as f:
    json.dump(l,f)
name

'code2_test_100.json'